# SuperMimic on Colab

One-time setup:
- Runtime, change runtime type, pick a GPU.
- Colab Secrets (key icon on the left): add `WANDB_API_KEY` and allow notebook access.

Run the cells top to bottom. Everything lives in `/content` and is rebuilt every session: the Python environment and the motion files. Checkpoints are uploaded to W&B by the training run, so nothing needs Google Drive.

## 1. GPU check

In [ ]:
import subprocess

out = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
).stdout.strip()
print(out)
assert out, "No GPU found. Change the runtime type to a GPU."
driver = out.splitlines()[0].split(",")[1].strip()
assert int(driver.split(".")[0]) >= 580, (
    f"NVIDIA driver {driver} is too old. The pinned torch build (CUDA 13) needs driver 580 or newer."
)

## 2. Repos

Clones `mjlab_research` (`main`) and `supermimic` (`trust`) as siblings under `/content/supermimic_ws`. To pin a sweep to a fixed commit hash, detach HEAD to that hash after checking out `trust`.

In [ ]:
# No Google Drive: motions are rebuilt each session and checkpoints go to W&B.
STORE = "/content/store"
WS = "/content/supermimic_ws"

# Sibling clones: pyproject.toml resolves mjlab from ../mjlab_research.
!mkdir -p {WS} {STORE}/data {STORE}/eval {STORE}/trust
!if [ -d {WS}/mjlab_research ]; then git -C {WS}/mjlab_research pull --ff-only; else git clone --branch main https://github.com/xSurus/mjlab_research.git {WS}/mjlab_research; fi
!if [ -d {WS}/supermimic ]; then git -C {WS}/supermimic fetch origin && git -C {WS}/supermimic checkout -B trust origin/trust; else git clone --branch trust https://github.com/xSurus/supermimic.git {WS}/supermimic; fi
# Pin to a fixed commit hash if needed so every run in a sweep uses the exact same code:
# !git -C {WS}/supermimic checkout --detach <hash>
!git -C {WS}/supermimic rev-parse --short HEAD && git -C {WS}/mjlab_research rev-parse --short HEAD
# data/ is not in .gitignore, so it stays outside the repo and cells use absolute paths.

## 3. Environment

In [ ]:
!pip install -q uv
!cd {WS}/supermimic && env -u PYTHONPATH uv sync --locked --group dev
out = !cd {WS}/supermimic && env -u PYTHONPATH uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
print("\n".join(out))
assert out and out[-1].endswith("True"), "The uv environment is missing or torch cannot see the GPU. Read the uv sync output above."

## 4. Weights & Biases

In [ ]:
import getpass
import os

try:
    from google.colab import userdata

    key = userdata.get("WANDB_API_KEY")
except Exception:
    key = None
os.environ["WANDB_API_KEY"] = key or getpass.getpass("W&B API key: ")

## 5. Tests

In [ ]:
!cd {WS}/supermimic && MUJOCO_GL=disable env -u PYTHONPATH uv run pytest -q -rs

## 6. Motions

Downloads the 40 LAFAN1 G1 retargeted `.npz` clips from W&B into `/content/store/data/npz/lafan1`.

In [ ]:
!cd {WS}/supermimic && env -u PYTHONPATH uv run wandb artifact get surus/supermimic/lafan1-g1-npz:latest --root {STORE}/data/npz/lafan1

## 7. Score finished `walk1-v2` runs

Scores the latest checkpoint of every finished `walk1-v2-<arm>-s<seed>` run (`base`, `fresh`, `dmo`, `base-cap`, `dmo-cap`, seeds 1–3) on `walk1_subject1.npz`, deterministic policy, randomization off. Results go to `{STORE}/eval/<run name>.json`. If a name was used twice, the newest finished run wins.

In [ ]:
import json
import os
import subprocess

names = [
    f"walk1-v2-{a}-s{s}"
    for a in ("base", "fresh", "dmo", "base-cap", "dmo-cap")
    for s in (1, 2, 3)
]
env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
repo = f"{WS}/supermimic"
os.makedirs(f"{STORE}/eval", exist_ok=True)
lookup = (
    "import json, sys, wandb; runs = wandb.Api().runs('surus/supermimic', {'state': 'finished'}, order='+created_at');"
    "print(json.dumps({r.name: r.id for r in runs if r.name in sys.argv[1:]}))"
)
out = subprocess.run(
    ["uv", "run", "python", "-c", lookup, *names],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True,
    check=True,
)
ids = json.loads(out.stdout.strip().splitlines()[-1])
print(f"found {len(ids)} finished runs")
scores = {}
for name in names:
    if name not in ids:
        continue
    out_file = f"{STORE}/eval/{name}.json"
    if not os.path.exists(out_file):
        res = subprocess.run(
            [
                "uv",
                "run",
                "src/supermimic/scripts/evaluate.py",
                "SuperMimic-GenCoord-G1",
                "--wandb-path",
                f"surus/supermimic/{ids[name]}",
                "--motion-file",
                f"{STORE}/data/npz/lafan1/walk1_subject1.npz",
                "--output-file",
                out_file,
            ],
            cwd=repo,
            env=env,
            capture_output=True,
            text=True,
        )
        if not os.path.exists(out_file):
            print(f"{name}: evaluation failed\n{res.stdout[-2000:]}\n{res.stderr[-2000:]}")
            continue
    with open(out_file) as f:
        r = json.load(f)
    scores[name] = (r["mean_normalized_return"], r["failed_clips"])
for name, (ret, fell) in scores.items():
    print(f"{name:20s} return={ret:.4f} fell={fell}")

## 8. Gradient trust (`N = 2048`, GPU, Step 4 + mixed-age + no-boot)

How well each gradient predicts the real simulator, at the final checkpoint of finished runs: imagined with model states (SuperMimic) and replayed through simulator states (DMO), from the same `2048` valid starts (`--chunk-size 512` for T4 VRAM safety). Runs a single sweep per checkpoint with:
- `--warmup-schedule 0 5 25 100`: mixes RSI-perturbed early starts (`0`, `5`) with steady-state starts (`25`, `100`) so the gradient tail is not filtered out.
- `--eps-values 0 0.005 0.015 0.05 --ls-iterations 100`: evaluates the float32 noise floor (`0`) and all perturbation sizes on identical starts and gradients without linesearch overflow.
- Step 4 `feet_ground_contact` phase bins (`flight`, `single_support`, `double_support`, `impact`), symmetric `paired (model bands)` and `paired (sim bands)`, `Spearman(G_m, G_s)`, tail `Jaccard` overlap, and `--check-noboot` (`||g_V||/||g||` + exact no-bootstrap trust).

Results (JSON + raw `.pt`) go to `{STORE}/trust/`.

In [ ]:
import json
import os
import subprocess

env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
repo = f"{WS}/supermimic"
os.makedirs(f"{STORE}/trust", exist_ok=True)
lookup = (
    "import json, sys, wandb; runs = wandb.Api().runs('surus/supermimic', {'state': 'finished'}, order='+created_at');"
    "print(json.dumps({r.name: r.id for r in runs if r.name in sys.argv[1:]}))"
)
TRUST_RUNS = ["walk1-v2-base-s1", "walk1-v2-dmo-cap-s1"]
out = subprocess.run(
    ["uv", "run", "python", "-c", lookup, *TRUST_RUNS],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True,
    check=True,
)
ids = json.loads(out.stdout.strip().splitlines()[-1])
for name in TRUST_RUNS:
    if name not in ids:
        print(f"{name}: no finished run")
        continue
    !cd {repo} && env -u PYTHONPATH uv run src/supermimic/scripts/grad_trust.py SuperMimic-GenCoord-G1 --wandb-path surus/supermimic/{ids[name]} --motion-file {STORE}/data/npz/lafan1/walk1_subject1.npz --num-starts 2048 --chunk-size 512 --warmup-schedule 0 5 25 100 --eps-values 0 0.005 0.015 0.05 --ls-iterations 100 --check-noboot --output-file {STORE}/trust/{name}-n2048-step4.json

## 9. On-policy DMO causal test (`dmo_hold=1, H=16, cap=10` — A100/H100 only)

**Do not run this on a T4.** At `dmo_hold=1, H=16` (`1` policy update per `17` env steps), keeping default `--runner.replay-ratio 16` over `5,000` steps only gives `294` policy updates against `998` WM updates (`3.40x` excess WM compute = ~90% of T4 wall clock, under-budgeted for convergence).

On an A100/H100, `--runner.replay-ratio 5 --runner.max-iterations 17000` at `2048` envs matches WM and policy updates (`1,062` WM vs `1,000` policy updates = `1.06x` parity, ~20 min each on A100; at `4096` envs use integer `--runner.replay-ratio 2` or `3`). Comparing `sim` against `model-fresh` isolates whether `dmo_hold=0` staleness caused the `walk1-v2` DMO lag cleanly at equal update counts.

In [ ]:
HOLD_COMMON = (
    f"--motion-file {STORE}/data/npz/lafan1/walk1_subject1.npz "
    "--num-envs 2048 --runner.policy.batch-size 1024 --runner.max-iterations 17000 "
    "--runner.replay-ratio 5 --runner.dmo-hold 1 --runner.policy.horizon 16 "
    "--runner.policy.rollout-cap 10 --env.seed 1"
)
HOLD_ARMS = {
    "walk1-v2-dmo-hold1-h16-rr5-s1": "--runner.states-from sim",
    "walk1-v2-fresh-hold1-h16-rr5-s1": "--runner.states-from model-fresh",
}
for run_name, flags in HOLD_ARMS.items():  # noqa: B007
    !cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/train.py SuperMimic-GenCoord-G1 {HOLD_COMMON} {flags} --runner.run-name {run_name}

In [ ]:
# Score the on-policy hold=1, H=16 pair on walk1 (deterministic, nominal physics).
import json
import os
import subprocess

env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
repo = f"{WS}/supermimic"
os.makedirs(f"{STORE}/eval", exist_ok=True)
lookup = (
    "import json, sys, wandb; runs = wandb.Api().runs('surus/supermimic', {'state': 'finished'}, order='+created_at');"
    "print(json.dumps({r.name: r.id for r in runs if r.name in sys.argv[1:]}))"
)
hold_names = ["walk1-v2-dmo-hold1-h16-rr5-s1", "walk1-v2-fresh-hold1-h16-rr5-s1"]
out = subprocess.run(
    ["uv", "run", "python", "-c", lookup, *hold_names],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True,
    check=True,
)
ids = json.loads(out.stdout.strip().splitlines()[-1])
for name in hold_names:
    if name not in ids:
        print(f"{name}: no finished run")
        continue
    out_file = f"{STORE}/eval/{name}.json"
    !cd {repo} && env -u PYTHONPATH uv run src/supermimic/scripts/evaluate.py SuperMimic-GenCoord-G1 --wandb-path surus/supermimic/{ids[name]} --motion-file {STORE}/data/npz/lafan1/walk1_subject1.npz --output-file {out_file}
    with open(out_file) as f:
        r = json.load(f)
    print(f"{name:30s} return={r['mean_normalized_return']:.4f} fell={r['failed_clips']}")

## 10. Full 40-clip LAFAN1 run & evaluation

The paper's LAFAN1 run: all 40 clips, 4096 envs, policy batch 4096, one NVIDIA H100 (80 GB). It reached a normalized return of 0.82 in evaluation within 1.2 h on that GPU (section 4.1). A T4 cannot hold this. In Runtime, change runtime type, pick H100 if it is offered, else A100. If an A100 runs out of memory, halve `--runner.policy.batch-size` and say so when comparing to the paper.

The loop runs until `--runner.max-iterations` or until you stop it. Checkpoints go to W&B every 1000 steps, so stopping loses nothing saved. Watch `Perf/` for the step rate and `Train/mean_reward`, which is not the paper's normalized return.

In [ ]:
import glob

clips = sorted(glob.glob(f"{STORE}/data/npz/lafan1/*.npz"))
assert len(clips) == 40, f"expected the 40 LAFAN1 clips, found {len(clips)}. Run section 6."
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/train.py SuperMimic-GenCoord-G1 --motion-file '{STORE}/data/npz/lafan1/*.npz' --num-envs 4096 --runner.max-iterations 200000 --runner.run-name lafan40-4096

### 10b. Evaluate a checkpoint

Scores a run's checkpoint the way the paper scores LAFAN1: each of the 40 clips is played once from its first frame with the deterministic policy, and the score is the episode's reward sum divided by the most it could be (section 4). It prints one line per clip, the mean normalized return and the number of clips where the robot fell. The paper's target is 0.82.

Set `RUN` to the training run (`entity/project/run_id`). `--checkpoint` takes a model artifact version such as `v3`; the default is the latest one. Running this while a training run uses the same GPU slows that run and skews its timings.

In [ ]:
RUN = "surus/supermimic/<run id>"
!cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/evaluate.py SuperMimic-GenCoord-G1 --wandb-path {RUN} --motion-file '{STORE}/data/npz/lafan1/*.npz' --output-file {STORE}/eval/latest.json

## 11. Play

In [ ]:
# Replace the placeholders with the W&B entity and run id.
# !cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/play.py SuperMimic-GenCoord-G1 --wandb-path <entity>/supermimic/<run id>